# NeuralMath — De las matemáticas a Python

Implementaremos ahora, utilizando únicamente NumPy , la propagación hacia delante, la regla de la cadena y las actualizaciones estocásticas derivadas en la Sección 7 [ 4 ] . Para obtener una frontera de decisión no lineal ampliamos la capa oculta de tres a doce neuronas ReLU. La red \(2\) – \(12\) – \(1\) contiene \(12(2+1)+(12+1)=49\) parámetros, pero sigue exactamente las mismas ecuaciones.

**NumPy · 2–12–1 · 64 + 8 · 6000 · η = 0.05**

Construcción del conjunto de datos sintéticos


## 01 — Preparación del entorno


In [ ]:
"""Ejemplo didáctico: clasificación de frutas con una red 2-12-1.
Dependencias: numpy y matplotlib. Ejecución: python neural_network.py.
"""
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D




## 02 — Activación ReLU


In [ ]:
def relu(s):
    """Activación de la capa oculta: ReLU(s) = max(0,s)."""
    return np.maximum(0, s)




## 03 — Derivada computacional de ReLU


In [ ]:
def relu_derivative(s):
    # ReLU no es diferenciable en s=0; se fija su derivada computacional en 0.
    # Esta convención solo define el paso de retropropagación en dicho punto.
    return (s > 0).astype(float)




## 04 — Activación sigmoide


In [ ]:
def sigmoid(s):
    """Activación de salida, con valores entre 0 y 1."""
    return 1 / (1 + np.exp(-s))




## 05 — Curva auxiliar para generar los datos


In [ ]:
def reference_curve(m):
    # Curva artificial (cm): desplazamiento, elevación gaussiana y ascenso logístico.
    # Las constantes se eligieron para construir una curva de referencia no lineal;
    # no son medidas experimentales ni parámetros ajustados.
    return (6.68 + 0.74*np.exp(-((m-155)/20)**2)
            + 1.50/(1 + np.exp(-(m-202)/5)))




## 06 — Datos sintéticos de entrenamiento y prueba ilustrativa


In [ ]:
def construct_data():
    # 16 masas en cada una de cuatro franjas: a1, a2, l1, l2.
    masses_1 = np.linspace(118, 212, 16)
    # Pequeños desplazamientos deterministas evitan masas idénticas.
    masses_2 = masses_1 + np.array([
        1.2, -1.1, 0.8, -1.3, 1.0, -0.8, 1.1, -1.0,
        1.3, -0.9, 0.9, -1.2, 1.0, -1.0, 0.8, -0.7
    ])
    k = np.arange(16)

    # Manzanas: dos franjas sobre la curva, con pequeñas variaciones en cm.
    a1 = np.column_stack([
        masses_1,
        reference_curve(masses_1) + 0.22 + 0.05*np.sin(0.8*k)
    ])
    a2 = np.column_stack([
        masses_2,
        reference_curve(masses_2) + 0.58 + 0.06*np.cos(0.45*k)
    ])

    # Naranjas: dos franjas bajo la curva de referencia.
    l1 = np.column_stack([
        masses_1,
        reference_curve(masses_1) - (0.22 + 0.04*np.cos(0.7*k))
    ])
    l2 = np.column_stack([
        masses_2,
        reference_curve(masses_2) - (0.55 + 0.05*np.sin(0.5*k))
    ])

    X_train = np.vstack([a1, a2, l1, l2])
    y_train = np.array([1.0]*32 + [0.0]*32)

    # Ocho ejemplos ilustrativos; no constituyen una prueba estadística amplia.
    X_test = np.array([
        [121.0, 7.20], [135.0, 7.45],
        [152.0, 7.75], [171.0, 7.32],
        [121.0, 6.28], [152.0, 6.86],
        [189.0, 6.48], [208.0, 7.55]
    ])
    y_test = np.array([1, 1, 1, 1, 0, 0, 0, 0], dtype=float)
    return X_train, y_train, X_test, y_test




## 07 — Predicciones de la propagación hacia delante


In [ ]:
def predict(X, mean, std, W1, b1, W2, b2):
    # x_j^* = (x_j - mu_j)/sigma_j, con estadísticas del conjunto de entrenamiento.
    Xn = (X - mean) / std
    # Paso hacia delante en notación matricial (una fila por fruta).
    h = relu(Xn @ W1.T + b1)
    return sigmoid(h @ W2 + b2)




## 08 — Entrenamiento por retropropagación y descenso del gradiente


In [ ]:
def train(Xn, y_train, eta=0.05, n_epochs=6000, seed=42):
    rng = np.random.default_rng(seed)
    # Los pesos aleatorios pequeños rompen la simetría entre neuronas;
    # los sesgos comienzan en cero. La escala 0.1 tiene fines didácticos.
    W1 = 0.1 * rng.standard_normal((12, 2))
    b1 = np.zeros(12)
    W2 = 0.1 * rng.standard_normal(12)
    b2 = 0.0
    history = []

    for epoch in range(n_epochs):
        # SGD: una actualización por fruta, con barajado en cada época.
        for i in rng.permutation(len(Xn)):
            x, target = Xn[i], y_train[i]

            # Paso hacia delante: ecuaciones de las capas oculta y de salida.
            z1 = W1 @ x + b1
            h = relu(z1)
            z2 = W2 @ h + b2
            y_hat = sigmoid(z2)

            # Retropropagación: delta2 = dE/dz2 (error cuadrático).
            delta2 = (y_hat - target) * y_hat * (1 - y_hat)
            dW2, db2 = delta2 * h, delta2
            # delta1_j = delta2 * W2_j * ReLU'(z1_j).
            delta1 = (delta2 * W2) * relu_derivative(z1)
            dW1 = np.outer(delta1, x)
            db1 = delta1

            # Descenso estocástico: theta <- theta - eta*dE/dtheta.
            # Todos los gradientes utilizan los valores ANTERIORES de los parámetros.
            W1 -= eta * dW1
            b1 -= eta * db1
            W2 -= eta * dW2
            b2 -= eta * db2

        # J: pérdida media de entrenamiento al final de cada época.
        h_all = relu(Xn @ W1.T + b1)
        y_pred = sigmoid(h_all @ W2 + b2)
        history.append(np.mean(0.5 * (y_train - y_pred)**2))

    return W1, b1, W2, b2, np.array(history)




## 09 — Representación de la frontera aprendida


In [ ]:
def plot_results(X_train, y_train, X_test, y_test,
                  mean, std, W1, b1, W2, b2, save_path=None):
    # Nivel y_hat=0.5: frontera de decisión APRENDIDA.
    masses = np.linspace(116, 214, 500)
    diameters = np.linspace(6.0, 8.85, 500)
    M, D = np.meshgrid(masses, diameters)
    grid = np.column_stack([M.ravel(), D.ravel()])
    P = predict(grid, mean, std, W1, b1, W2, b2).reshape(M.shape)

    fig, ax = plt.subplots(figsize=(9.2, 6.3))
    ax.contourf(M, D, P, levels=[0, 0.25, 0.50, 0.75, 1],
                colors=['#f3e0cb', '#ecd5c4', '#dbe7cf', '#cde3bf'],
                alpha=0.95)
    ax.contour(M, D, P, levels=[0.5], colors=['#5a1a8a'],
               linewidths=2.2)

    mask = y_train == 1.0
    ax.scatter(X_train[mask, 0], X_train[mask, 1], marker='o',
               s=48, facecolors='white', edgecolors='black', linewidths=1.1)
    mask = y_train == 0.0
    ax.scatter(X_train[mask, 0], X_train[mask, 1], marker='s',
               s=48, facecolors='white', edgecolors='black', linewidths=1.1)
    mask = y_test == 1.0
    ax.scatter(X_test[mask, 0], X_test[mask, 1], marker='^',
               s=92, color='#1f77b4')
    mask = y_test == 0.0
    ax.scatter(X_test[mask, 0], X_test[mask, 1], marker='x',
               s=110, color='#ff7f0e', linewidths=2.0)

    ax.set_xlabel('Masa (g)')
    ax.set_ylabel('Diámetro (cm)')
    ax.set_title('Salida de la red neuronal después del entrenamiento')
    handles = [
        Line2D([0], [0], marker='o', markersize=9, markerfacecolor='white',
               markeredgecolor='black', linestyle='None',
               label='Manzana - entrenamiento'),
        Line2D([0], [0], marker='s', markersize=9, markerfacecolor='white',
               markeredgecolor='black', linestyle='None',
               label='Naranja - entrenamiento'),
        Line2D([0], [0], marker='^', markersize=9, color='#1f77b4',
               linestyle='None', label='Manzana - prueba'),
        Line2D([0], [0], marker='x', markersize=10, color='#ff7f0e',
               linestyle='None', label='Naranja - prueba'),
        Line2D([0], [0], color='#5a1a8a', linewidth=2.2,
               label='Frontera aprendida')
    ]
    ax.legend(handles=handles, loc='center left',
              bbox_to_anchor=(1.02, 0.5), frameon=True)
    fig.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=200, bbox_inches='tight')
    return fig, ax




## 10 — Ejecución del experimento


In [ ]:
if __name__ == '__main__':
    X_train, y_train, X_test, y_test = construct_data()
    mean, std = X_train.mean(axis=0), X_train.std(axis=0)
    Xn = (X_train - mean) / std
    W1, b1, W2, b2, history = train(Xn, y_train)
    p_train = predict(X_train, mean, std, W1, b1, W2, b2)
    p_test = predict(X_test, mean, std, W1, b1, W2, b2)
    print('Pérdida media final:', history[-1])
    print('Predicciones correctas (entrenamiento):', np.sum((p_train >= 0.5) == y_train), '/64')
    print('Predicciones correctas (prueba ilustrativa):',
          np.sum((p_test >= 0.5) == y_test), '/8')
    plot_results(X_train, y_train, X_test, y_test,
                 mean, std, W1, b1, W2, b2)
    plt.show()
